In [7]:
# DAY 2: MACHINE TRANSLATION - EN -> ES

!pip install transformers datasets torch sacrebleu sentencepiece accelerate -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
from sacrebleu import corpus_bleu
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print("DAY 2: MACHINE TRANSLATION")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. In Colab, go to Runtime > Change runtime "
          "type > GPU before fine-tuning MarianMT, or training will be slow.")


if device == "cpu":
    DATA_SAMPLE_SIZE = 1500
    SEQ2SEQ_EPOCHS = 8
    MARIAN_EPOCHS = 1
    BATCH_SIZE = 8
    print(f"Running in CPU-scaled mode: sample={DATA_SAMPLE_SIZE}, "
          f"seq2seq_epochs={SEQ2SEQ_EPOCHS}, marian_epochs={MARIAN_EPOCHS}, "
          f"batch_size={BATCH_SIZE}")
else:
    DATA_SAMPLE_SIZE = 6000
    SEQ2SEQ_EPOCHS = 15
    MARIAN_EPOCHS = 3
    BATCH_SIZE = 16

# loading dataset
raw = load_dataset("Helsinki-NLP/opus_books", "en-es")["train"]

pairs = raw["translation"]
en_sents = [p["en"].strip() for p in pairs]
es_sents = [p["es"].strip() for p in pairs]


filtered = [
    (en, es) for en, es in zip(en_sents, es_sents)
    if 3 <= len(en.split()) <= 30 and 3 <= len(es.split()) <= 30
]
print(f"Full corpus: {len(pairs)} pairs -> {len(filtered)} pairs after length filtering")


rng = np.random.default_rng(42)
SAMPLE_SIZE = min(DATA_SAMPLE_SIZE, len(filtered))
sample_idx = rng.choice(len(filtered), size=SAMPLE_SIZE, replace=False)
sampled = [filtered[i] for i in sample_idx]

en_sents = [s[0] for s in sampled]
es_sents = [s[1] for s in sampled]

split = int(len(en_sents) * 0.9)
train_sources, test_sources = en_sents[:split], en_sents[split:]
train_targets, test_targets = es_sents[:split], es_sents[split:]

print(f"Dataset: {len(en_sents)} sentences (train={len(train_sources)}, test={len(test_sources)})")
print(f"Language pair: en -> es")

print("\nSample pairs:")
for i in range(3):
    print(f"en: {train_sources[i]}")
    print(f"es: {train_targets[i]}")
    print()

# 2. MODEL 1: BASELINE (word-level dictionary, statistical/rule-based)

print("MODEL 1: BASELINE (Dictionary)")

class DictionaryTranslator:
    def __init__(self):
        self.word_dict = {}

    def build_dict(self, src_texts, tgt_texts):

        for src, tgt in zip(src_texts, tgt_texts):
            src_words = src.lower().split()
            tgt_words = tgt.lower().split()
            for s, t in zip(src_words, tgt_words):
                if s not in self.word_dict:
                    self.word_dict[s] = t
        print(f"Dictionary size: {len(self.word_dict)} words")

    def translate(self, text):
        words = text.lower().split()
        translated = [self.word_dict.get(word, word) for word in words]
        return ' '.join(translated)

baseline = DictionaryTranslator()
baseline.build_dict(train_sources, train_targets)

baseline_preds = [baseline.translate(src) for src in test_sources]
baseline_bleu = corpus_bleu(baseline_preds, [test_targets])
print(f"Baseline BLEU: {baseline_bleu.score:.2f}")

print("\nSample translations:")
for i in range(3):
    print(f"Source: {test_sources[i]}")
    print(f"Predicted: {baseline_preds[i]}")
    print(f"Reference: {test_targets[i]}")
    print()


# 3. MODEL 2: SEQ2SEQ FROM SCRATCH (GRU encoder-decoder, no attention)

print("MODEL 2: SEQ2SEQ (from scratch)")


def simple_tokenize(texts, vocab=None, max_vocab=8000):
    if vocab is None:
        vocab = {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
    tokenized = []
    for text in texts:
        tokens = text.lower().split()
        token_ids = []
        for token in tokens:
            if token not in vocab and len(vocab) < max_vocab:
                vocab[token] = len(vocab)
            token_ids.append(vocab.get(token, vocab['<UNK>']))
        tokenized.append(token_ids)
    return tokenized, vocab

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        output = self.fc(output)
        return output, hidden

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, embed_dim=256, hidden_dim=512):
        super(Seq2Seq, self).__init__()
        self.encoder = Encoder(src_vocab, embed_dim, hidden_dim)
        self.decoder = Decoder(tgt_vocab, embed_dim, hidden_dim)

    def forward(self, src, tgt):
        encoder_outputs, hidden = self.encoder(src)
        outputs, _ = self.decoder(tgt[:, :-1], hidden)
        return outputs

def pad_sequences(sequences, max_len):
    padded = []
    for seq in sequences:
        if len(seq) > max_len:
            seq = seq[:max_len]
        padded.append(seq + [0] * (max_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)

print("Preparing data...")

src_tokenized, src_vocab = simple_tokenize(train_sources)
tgt_tokenized, tgt_vocab = simple_tokenize(
    ['<BOS> ' + t + ' <EOS>' for t in train_targets]
)

max_len = 32
src_padded = pad_sequences(src_tokenized, max_len)
tgt_padded = pad_sequences(tgt_tokenized, max_len)

train_dataset = TensorDataset(src_padded, tgt_padded)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

seq2seq_model = Seq2Seq(len(src_vocab), len(tgt_vocab)).to(device)
optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Training Seq2Seq...")
N_EPOCHS = SEQ2SEQ_EPOCHS
for epoch in range(N_EPOCHS):
    total_loss = 0
    for batch_src, batch_tgt in train_loader:
        batch_src, batch_tgt = batch_src.to(device), batch_tgt.to(device)
        optimizer.zero_grad()
        output = seq2seq_model(batch_src, batch_tgt)
        loss = criterion(output.reshape(-1, len(tgt_vocab)), batch_tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seq2seq_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS}: Loss = {total_loss/len(train_loader):.4f}")

print("Translating with Seq2Seq...")
seq2seq_model.eval()
seq2seq_preds = []
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}
bos_id, eos_id = tgt_vocab['<BOS>'], tgt_vocab['<EOS>']

for src in test_sources:
    tokens = src.lower().split()
    src_ids = [src_vocab.get(t, src_vocab['<UNK>']) for t in tokens]
    if len(src_ids) > max_len:
        src_ids = src_ids[:max_len]
    src_tensor = torch.tensor([src_ids + [0] * (max_len - len(src_ids))]).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = seq2seq_model.encoder(src_tensor)
        pred_tokens = []
        decoder_input = torch.tensor([[bos_id]]).to(device)
        for _ in range(max_len):
            output, hidden = seq2seq_model.decoder(decoder_input, hidden)
            next_token = output.argmax(2).item()
            if next_token == eos_id:
                break
            pred_tokens.append(next_token)
            decoder_input = torch.tensor([[next_token]]).to(device)

    pred_words = [inv_tgt_vocab.get(t, '<UNK>') for t in pred_tokens]
    seq2seq_preds.append(' '.join(pred_words) if pred_words else '<empty>')

seq2seq_bleu = corpus_bleu(seq2seq_preds, [test_targets])
print(f"Seq2Seq BLEU: {seq2seq_bleu.score:.2f}")

print("\nSample Seq2Seq translations:")
for i in range(3):
    print(f"Source: {test_sources[i]}")
    print(f"Predicted: {seq2seq_preds[i]}")
    print(f"Reference: {test_targets[i]}")
    print()

# 4. MODEL 3: FINE-TUNED PRETRAINED TRANSFORMER (MarianMT)



print("MODEL 3: FINE-TUNED PRETRAINED TRANSFORMER (MarianMT)")


model_name = "Helsinki-NLP/opus-mt-en-es"
print(f"Loading {model_name}...")

marian_tokenizer = AutoTokenizer.from_pretrained(model_name)
marian_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# --- BLEU BEFORE fine-tuning, for comparison ---
def marian_translate(texts, batch_size=BATCH_SIZE, max_length=128):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = marian_tokenizer(batch, return_tensors="pt", padding=True,
                                   truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            outputs = marian_model.generate(**inputs, max_length=max_length)
        preds.extend(marian_tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

print("\nTranslating with pretrained (not yet fine-tuned) MarianMT...")
marian_preds_before = marian_translate(test_sources)
marian_bleu_before = corpus_bleu(marian_preds_before, [test_targets])
print(f"MarianMT BLEU (before fine-tuning): {marian_bleu_before.score:.2f}")

# --- Fine-tune MarianMT on the training split ---
from datasets import Dataset

train_ds = Dataset.from_dict({'en': train_sources, 'es': train_targets})
test_ds = Dataset.from_dict({'en': test_sources, 'es': test_targets})

def preprocess(batch):
    model_inputs = marian_tokenizer(batch['en'], truncation=True, max_length=128)
    labels = marian_tokenizer(text_target=batch['es'], truncation=True, max_length=128)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=['en', 'es'])
test_ds = test_ds.map(preprocess, batched=True, remove_columns=['en', 'es'])

collator = DataCollatorForSeq2Seq(tokenizer=marian_tokenizer, model=marian_model)

def compute_bleu(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, marian_tokenizer.pad_token_id)
    decoded_preds = marian_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, marian_tokenizer.pad_token_id)
    decoded_labels = marian_tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = corpus_bleu(decoded_preds, [decoded_labels])
    return {'bleu': bleu.score}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/marian_ckpt",
    num_train_epochs=MARIAN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    predict_with_generate=True,
    generation_max_length=128,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=marian_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_bleu,
)

print("\nFine-tuning MarianMT...")
trainer.train()

print("\nTranslating with fine-tuned MarianMT...")
marian_preds_after = marian_translate(test_sources)
marian_bleu_after = corpus_bleu(marian_preds_after, [test_targets])
print(f"MarianMT BLEU (after fine-tuning): {marian_bleu_after.score:.2f}")

print("\nSample translations (fine-tuned):")
for i in range(3):
    print(f"Source: {test_sources[i]}")
    print(f"Predicted: {marian_preds_after[i]}")
    print(f"Reference: {test_targets[i]}")
    print()


# 5. RESULTS

print("RESULTS")

results = {
    'Model': [
        'Baseline (Dictionary)',
        'Seq2Seq (from scratch)',
        'MarianMT (pretrained, not fine-tuned)',
        'MarianMT (fine-tuned)'
    ],
    'BLEU Score': [
        baseline_bleu.score,
        seq2seq_bleu.score,
        marian_bleu_before.score,
        marian_bleu_after.score
    ]
}

results_df = pd.DataFrame(results).sort_values('BLEU Score', ascending=False)
print(results_df.to_string(index=False))

results_df.to_csv('/content/drive/MyDrive/day2_results.csv', index=False)
print("\nResults saved to: /content/drive/MyDrive/day2_results.csv")

predictions_df = pd.DataFrame({
    'source': test_sources,
    'reference': test_targets,
    'baseline_pred': baseline_preds,
    'seq2seq_pred': seq2seq_preds,
    'marian_before_pred': marian_preds_before,
    'marian_after_pred': marian_preds_after,
})
predictions_df.to_csv('/content/drive/MyDrive/day2_predictions.csv', index=False)
print("Predictions saved to: /content/drive/MyDrive/day2_predictions.csv")


best_row = results_df.iloc[0]
print(f"Best Model: {best_row['Model']}")
print(f"Best BLEU: {best_row['BLEU Score']:.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DAY 2: MACHINE TRANSLATION

Device: cpu
Running in CPU-scaled mode: sample=1500, seq2seq_epochs=8, marian_epochs=1, batch_size=8
Full corpus: 93470 pairs -> 64269 pairs after length filtering
Dataset: 1500 sentences (train=1350, test=150)
Language pair: en -> es

Sample pairs:
en: The production of these their first tools was hailed as a triumph.
es: La conquista de esta primera herramienta fue saludada como un triunfo; conquista preciosa, en efecto, y hecha muy oportunamente.

en: Then they looked at one another.
es: Luego se miraron.

en: Yes, a thicket of dead trees! Trees without leaves, without sap, turned to stone by the action of the waters, and crowned here and there by gigantic pines.
es: Sí, una espesura de árboles muertos, sin hojas, sin savia, árboles mineralizados por la acción del agua y de entre los que sobresalían aquí y allá algunos pinos gig

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


Translating with pretrained (not yet fine-tuned) MarianMT...
MarianMT BLEU (before fine-tuning): 19.16


Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


Fine-tuning MarianMT...


Epoch,Training Loss,Validation Loss,Bleu
1,2.179213,2.079796,19.695596



Translating with fine-tuned MarianMT...
MarianMT BLEU (after fine-tuning): 19.70

Sample translations (fine-tuned):
Source: She had finished her breakfast, so I permitted her to give a specimen of her accomplishments.
Predicted: Había terminado de desayunar, así que le permití dar una muestra de sus logros.
Reference: El desayuno había concluido y yo le permití que me diera una muestra de sus habilidades.

Source: "Hey, Herbert! how capital it sounds!
Predicted: -¡Eh, Harbert! ¡Qué bien suena!
Reference: –¡Harbert, esto marcha!

Source: "The first was from Pondicherry, the second from Dundee, and the third from London."
Predicted: El primero era de Pondicherry, el segundo de Dundee, y el tercero de Londres.
Reference: ––El primero era de Pondicherry, el segundo de Dundee, y el tercero de Londres.

RESULTS
                                Model  BLEU Score
                MarianMT (fine-tuned)   19.704662
MarianMT (pretrained, not fine-tuned)   19.163167
                Baseline (Dictio